In [ ]:
# Install the required libraries
!pip -q install pypdf openai numpy

# Pinecone is optional. Uncomment the next line only if you want hosted vector storage.
# !pip -q install pinecone


# Speak with PDF Assignment

**Assignment type:** RAG mini-app with three required functions  
**File to submit:** `pdf_ai.ipynb` (the notebook produced from this spec)  
**Language:** Python 3.x (Codio / Jupyter)  
**Time estimate:** 2–3 hours

---

## Objective

Build a **“Speak with PDF” RAG pipeline** that runs inside a Jupyter notebook and is compatible with **NBGrader**. You will implement **three functions**:

1) `parse_pdf_to_chunks(pdf_path, chunk_size=800, overlap=120) -> List[str]`  
2) `embed_and_index(chunks, use_pinecone=True, model="text-embedding-3-small") -> dict`  
3) `answer_query(question, index_obj, top_k=3, gpt_model="gpt-4o-mini") -> str`

**Grading will use OpenAI keys.** Learners must provide valid `OPENAI_API_KEY` (and Pinecone credentials if using Pinecone).

> **OpenRouter adaptation:** This completed version uses the OpenRouter OpenAI-compatible endpoint. Add `OPENROUTER_API_KEY` to Colab Secrets. The assignment's original text mentions OpenAI keys, but your course instructions specifically require OpenRouter. The notebook retains the required function names and default signatures for grading compatibility.


---

## What you will build

A tiny RAG system that:
- loads a PDF,
- extracts and **chunks** text,
- builds a **vector index** from OpenAI embeddings (Pinecone recommended; in‑memory/FAISS acceptable),
- answers a natural-language query by retrieving top‑k relevant chunks and generating an answer with an OpenAI chat model.

This mirrors production flow (PDF → chunks → embeddings → index → retrieval → answer).


---

## Tools & Libraries

- `openai` (for embeddings + chat completions)
- `pypdf` **or** `pdfplumber` (choose one) for text extraction
- Vector store: **Pinecone** (recommended) via `pinecone-client`; you may alternatively use **FAISS** or a simple in‑memory index for development
- `numpy`, `re`, `math`, `hashlib` (as needed)

> You may import additional libraries if helpful. Use your judgment.

**Environment variables expected (for grading):**
- `OPENAI_API_KEY`
- If using Pinecone: `PINECONE_API_KEY`, `PINECONE_ENV` (or project config), and an index name you create in notebook (e.g., `pdf-mini-app`).



---

## Function Requirements
You'll have to implment the following functionality down below

### 1) `parse_pdf_to_chunks(pdf_path, chunk_size=800, overlap=120) -> list[str]`

**Inputs**
- `pdf_path: str` — path to a readable PDF file (e.g., `data/sample.pdf`)
- `chunk_size: int` (default 800) — approximate number of characters per chunk
- `overlap: int` (default 120) — number of overlapping characters between adjacent chunks

**Output**
- `List[str]` — ordered, non-empty chunks of plain text. No chunk should exceed `chunk_size + 20` characters (allowing minor boundary variance).

**Behavioral constraints**
- Strip excessive whitespace and normalize newlines.
- Deterministic chunk boundaries given the same inputs.
- Apply overlap between chunks (except the first).

**Recommended approach**
1. Extract all PDF text into a single string using `pypdf` **or** `pdfplumber`.
2. Normalize whitespace.
3. Greedy chunk by character count with the requested overlap.



---

### 2) `embed_and_index(chunks, use_pinecone=True, model="text-embedding-3-small") -> dict`

**Purpose**  
Create embeddings with **OpenAI** and store them in a vector index (Pinecone preferred).

**Inputs**
- `chunks: List[str]` — the output of `parse_pdf_to_chunks`
- `use_pinecone: bool` — if `True`, upsert to Pinecone; if `False`, store locally (FAISS or simple in‑memory array search)
- `model: str` — OpenAI embedding model name (e.g., `"text-embedding-3-small"`)

**Output**
- `dict` with fields (minimum):
  - `"chunks"`: `List[str]`
  - `"index_backend"`: `"pinecone"` or `"local"`
  - `"index_ref"`: backend-specific handle (e.g., Pinecone index name / namespace, or local matrix + ids)
  - `"embed_model"`: embedding model used
  - `"dim"`: embedding dimensionality

**Notes**
- When `use_pinecone=True`: create (if needed) and upsert vectors with stable IDs (e.g., `doc-0`, `doc-1`, …). Keep a mapping from ID → original chunk text in the dict.
- When `use_pinecone=False`: store embeddings in memory (e.g., NumPy matrix) along with their chunk IDs for cosine‑similarity search.



---

### 3) `answer_query(question, index_obj, top_k=3, gpt_model="gpt-4o-mini") -> str`

**Purpose**  
Retrieve relevant chunks and generate an answer using an OpenAI chat model.

**Inputs**
- `question: str`
- `index_obj: dict` — the output from `embed_and_index`
- `top_k: int` — number of chunks to retrieve
- `gpt_model: str` — OpenAI chat model (e.g., `"gpt-4o-mini"`)

**Output**
- `str` — the final answer.

**Required behavior**
- Embed the **question** using the **same embedding model** used to build the index.
- Retrieve **top_k** chunks by cosine similarity (Pinecone query or local similarity search).
- Construct a concise prompt:
  - System: “You are a helpful assistant. Use ONLY the provided context to answer. If not answerable from context, respond: ‘I don’t know based on the document.’”
  - User: include the `question` and a `Context:` section concatenating the retrieved chunks (with simple separators).
- Call the OpenAI chat completion endpoint and **return the assistant’s message string**.



In [ ]:
# This cell may be run by the grader
import os, re, math, hashlib
import numpy as np

In [1]:
# Secure OpenRouter configuration
# Recommended: add OPENROUTER_API_KEY in Google Colab > Secrets (key icon).
# Do not paste your key directly into this notebook or commit it to GitHub.

import os

try:
    from google.colab import userdata
    _secret_key = userdata.get("OPENROUTER_API_KEY")
except Exception:
    _secret_key = None

OPENROUTER_API_KEY = _secret_key or os.getenv("OPENROUTER_API_KEY")
if not OPENROUTER_API_KEY:
    raise RuntimeError(
        "OpenRouter API key not found. Add OPENROUTER_API_KEY to Colab Secrets "
        "and enable notebook access, or set it as an environment variable."
    )

# OpenRouter exposes an OpenAI-compatible API.
from openai import OpenAI

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
    default_headers={
        "HTTP-Referer": "https://colab.research.google.com/",
        "X-Title": "Speak with the PDF - RAG Assignment",
    },
)

# Model identifiers on OpenRouter. Change these if your account does not have access.
EMBEDDING_MODEL = "openai/text-embedding-3-small"
CHAT_MODEL = "openai/gpt-4o-mini"

print("OpenRouter client configured. API key is hidden.")

OpenRouter client configured. API key is hidden.


In [2]:
# Keep compatibility with the starter notebook's expected variable names.
# Pinecone is optional; local NumPy vector search is the default and needs no extra service.
OPENAI_API_KEY = OPENROUTER_API_KEY  # compatibility alias only; requests still go to OpenRouter
USE_PINECONE = bool(os.getenv("PINECONE_API_KEY"))
print("Vector backend:", "Pinecone (if installed/configured)" if USE_PINECONE else "Local NumPy")

Vector backend: Local NumPy


<center><h1>--------------- Student Code ---------------</h1></center>

### B. Implement `parse_pdf_to_chunks`  **(student solution cell)**

### Please Write your answer here

Delete the `raise NotImplementedError()` when you start writing your code.

In [ ]:
def parse_pdf_to_chunks(pdf_path, chunk_size=800, overlap=120):
    """
    Extract text from a PDF and split it into deterministic overlapping chunks.

    Args:
        pdf_path (str): Path to a readable PDF.
        chunk_size (int): Maximum target chunk length in characters.
        overlap (int): Number of characters shared by adjacent chunks.

    Returns:
        list[str]: Ordered, non-empty text chunks.
    """
    from pypdf import PdfReader

    if not isinstance(chunk_size, int) or chunk_size <= 0:
        raise ValueError("chunk_size must be a positive integer.")
    if not isinstance(overlap, int) or overlap < 0:
        raise ValueError("overlap must be a non-negative integer.")
    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size.")
    if not os.path.isfile(pdf_path):
        raise FileNotFoundError(f"PDF file not found: {pdf_path}")

    reader = PdfReader(pdf_path)
    page_texts = []
    for page in reader.pages:
        text = page.extract_text() or ""
        if text.strip():
            page_texts.append(text)

    # Normalize whitespace while preserving readable word separation.
    full_text = " ".join(page_texts)
    full_text = re.sub(r"\s+", " ", full_text).strip()
    if not full_text:
        return []

    # Fixed character windows make chunk boundaries deterministic and ensure
    # every chunk stays within chunk_size. Adjacent chunks share `overlap` chars.
    chunks = []
    step = chunk_size - overlap
    for start in range(0, len(full_text), step):
        chunk = full_text[start:start + chunk_size].strip()
        if chunk:
            chunks.append(chunk)
        if start + chunk_size >= len(full_text):
            break

    return chunks

---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `parse_pdf_to_chunks`.
 -  If the function is not found, you answer will be rejected by the system

### C. Implement `embed_and_index`  **(student solution cell)**

### Please Write your answer here

Delete the `raise NotImplementedError()` when you start writing your code.

In [ ]:
def embed_and_index(chunks, use_pinecone=True, model="text-embedding-3-small"):
    """
    Embed text chunks through OpenRouter and create a vector index.

    Pinecone is used only when requested, its API key is present, and the
    Pinecone package is installed. Otherwise a local NumPy index is returned.

    Returns a dictionary containing chunks, backend, index handle, model,
    dimension, stable IDs, and the embedding matrix for local search.
    """
    if not chunks:
        raise ValueError("chunks must contain at least one non-empty text chunk.")

    clean_chunks = [str(chunk).strip() for chunk in chunks if str(chunk).strip()]
    if not clean_chunks:
        raise ValueError("No non-empty chunks were supplied.")

    # OpenRouter model slugs are namespaced; accept the starter's bare model name.
    embed_model = model if "/" in model else f"openai/{model}"

    response = client.embeddings.create(
        model=embed_model,
        input=clean_chunks
    )
    # Preserve the provider's input order.
    embeddings = np.asarray(
        [item.embedding for item in sorted(response.data, key=lambda x: x.index)],
        dtype=np.float32
    )
    if embeddings.ndim != 2 or embeddings.shape[0] != len(clean_chunks):
        raise RuntimeError("Embedding API returned an unexpected embedding shape.")

    ids = [f"doc-{i}" for i in range(len(clean_chunks))]
    index_obj = {
        "chunks": clean_chunks,
        "index_backend": "local",
        "index_ref": None,
        "embed_model": embed_model,
        "dim": int(embeddings.shape[1]),
        "ids": ids,
        "id_to_chunk": dict(zip(ids, clean_chunks)),
        "embeddings": embeddings,
    }

    # Optional Pinecone path. Local NumPy remains the reliable fallback.
    if use_pinecone and os.getenv("PINECONE_API_KEY"):
        try:
            from pinecone import Pinecone, ServerlessSpec
            pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
            index_name = os.getenv("PINECONE_INDEX_NAME", "pdf-mini-app").lower().replace("_", "-")

            existing = [item["name"] for item in pc.list_indexes()]
            if index_name not in existing:
                pc.create_index(
                    name=index_name,
                    dimension=index_obj["dim"],
                    metric="cosine",
                    spec=ServerlessSpec(
                        cloud=os.getenv("PINECONE_CLOUD", "aws"),
                        region=os.getenv("PINECONE_REGION", "us-east-1")
                    )
                )
            pinecone_index = pc.Index(index_name)
            vectors = [
                {
                    "id": ids[i],
                    "values": embeddings[i].astype(float).tolist(),
                    "metadata": {"chunk": clean_chunks[i], "order": i}
                }
                for i in range(len(clean_chunks))
            ]
            # Upsert in manageable batches.
            for start in range(0, len(vectors), 100):
                pinecone_index.upsert(vectors=vectors[start:start + 100])
            index_obj.update({
                "index_backend": "pinecone",
                "index_ref": index_name,
                "pinecone_index": pinecone_index,
                "pc_client": pc,
            })
        except Exception as exc:
            print(f"Pinecone setup failed; using local NumPy index instead. Reason: {exc}")

    return index_obj

---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `embed_and_index`.
 -  If the function is not found, you answer will be rejected by the system

### D. Implement `answer_query`  **(student solution cell)**

### Please Write your answer here

Delete the `raise NotImplementedError()` when you start writing your code.

In [ ]:
def answer_query(question, index_obj, top_k=3, gpt_model="gpt-4o-mini"):
    """
    Retrieve the most relevant chunks and ask an OpenRouter chat model
    to answer using only the retrieved PDF context.
    """
    if not isinstance(question, str) or not question.strip():
        raise ValueError("question must be a non-empty string.")
    if not index_obj or not index_obj.get("chunks"):
        raise ValueError("index_obj is empty or invalid.")
    if not isinstance(top_k, int) or top_k <= 0:
        raise ValueError("top_k must be a positive integer.")

    embed_model = index_obj["embed_model"]
    q_response = client.embeddings.create(
        model=embed_model,
        input=[question.strip()]
    )
    query_vector = np.asarray(q_response.data[0].embedding, dtype=np.float32)

    if index_obj["index_backend"] == "pinecone":
        result = index_obj["pinecone_index"].query(
            vector=query_vector.astype(float).tolist(),
            top_k=min(top_k, len(index_obj["chunks"])),
            include_metadata=True
        )
        matches = sorted(
            result.matches,
            key=lambda match: (-float(match.score or 0.0), str(match.id))
        )
        retrieved_chunks = [
            match.metadata.get("chunk", "")
            for match in matches if match.metadata and match.metadata.get("chunk")
        ]
    else:
        matrix = np.asarray(index_obj["embeddings"], dtype=np.float32)
        # Cosine similarity; zero vectors are safely assigned similarity 0.
        matrix_norms = np.linalg.norm(matrix, axis=1)
        query_norm = np.linalg.norm(query_vector)
        denom = matrix_norms * query_norm
        scores = np.divide(
            matrix @ query_vector,
            denom,
            out=np.zeros_like(matrix_norms, dtype=np.float32),
            where=denom != 0
        )
        # Stable tie-break: original chunk order.
        ranked_indices = sorted(
            range(len(scores)),
            key=lambda i: (-float(scores[i]), i)
        )[:min(top_k, len(scores))]
        retrieved_chunks = [index_obj["chunks"][i] for i in ranked_indices]

    context = "\n\n---\n\n".join(retrieved_chunks)
    if not context.strip():
        return "I don’t know based on the document."

    chat_model = gpt_model
    if "/" not in chat_model:
        chat_model = f"openai/{chat_model}"

    completion = client.chat.completions.create(
        model=chat_model,
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a helpful assistant. Use ONLY the provided context to answer. "
                    "If the answer is not supported by the context, respond exactly: "
                    "I don’t know based on the document."
                )
            },
            {
                "role": "user",
                "content": f"Question: {question.strip()}\n\nContext:\n{context}"
            }
        ]
    )
    return (completion.choices[0].message.content or "").strip()

---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `answer_query`.
 -  If the function is not found, you answer will be rejected by the system


### E. (Optional) Demo cell (ungraded)
Provide a small demo that:
- Loads `data/sample.pdf`
- Runs all three functions
- Prints an example answer



In [ ]:
# Upload a PDF from your computer in Google Colab.
# After upload, set PDF_PATH to the uploaded filename.
from google.colab import files

uploaded = files.upload()
if uploaded:
    PDF_PATH = next(iter(uploaded))
    print(f"Uploaded: {PDF_PATH}")
else:
    PDF_PATH = None
    print("No file uploaded yet.")

In [ ]:
# Build the document index.
# This uses local NumPy search by default. Set use_pinecone=True only if
# PINECONE_API_KEY is configured and you want to use Pinecone.
if PDF_PATH:
    chunks = parse_pdf_to_chunks(PDF_PATH, chunk_size=800, overlap=120)
    print(f"Extracted {len(chunks)} chunks.")
    index_obj = embed_and_index(chunks, use_pinecone=False, model=EMBEDDING_MODEL)
    print("Index backend:", index_obj["index_backend"])
    print("Embedding dimension:", index_obj["dim"])
else:
    print("Upload a PDF in the previous cell before building the index.")

In [ ]:
# Ask a question about your uploaded PDF.
if PDF_PATH and "index_obj" in globals():
    question = "What are the main ideas discussed in this document?"
    print(answer_query(question, index_obj, top_k=3, gpt_model=CHAT_MODEL))
else:
    print("Upload a PDF and build the index first.")

In [ ]:
# Comparative playtest: enter three questions that are specific to your PDF.
TEST_QUESTIONS = [
    "What is the first important concept explained in the document?",
    "What process, method, or framework does the document describe?",
    "What limitation, conclusion, or recommendation does the document mention?"
]

def vanilla_llm_answer(question, model=CHAT_MODEL):
    """Answer without PDF retrieval, for comparison with the RAG answer."""
    response = client.chat.completions.create(
        model=model,
        temperature=0,
        messages=[
            {"role": "system", "content": "Answer the user's question using your general knowledge. Do not claim to have read a document."},
            {"role": "user", "content": question}
        ]
    )
    return (response.choices[0].message.content or "").strip()

if PDF_PATH and "index_obj" in globals():
    comparison_results = []
    for i, q in enumerate(TEST_QUESTIONS, start=1):
        vanilla = vanilla_llm_answer(q)
        rag = answer_query(q, index_obj, top_k=3, gpt_model=CHAT_MODEL)
        comparison_results.append({"question": q, "vanilla": vanilla, "rag": rag})
        print(f"\n{'='*80}\nQUESTION {i}: {q}\n\nVANILLA LLM:\n{vanilla}\n\nRAG ANSWER:\n{rag}")
else:
    print("Run the PDF upload and indexing cells before comparative playtesting.")

In [ ]:
# Save your experiment results as a Markdown file if desired.
# Review the answers above and complete the reflection section below using your
# actual observations. Do not report generated outputs you have not inspected.
if "comparison_results" in globals():
    print(f"Recorded {len(comparison_results)} question comparisons.")
    print("Review each pair for accuracy, hallucination, relevance, and completeness.")
else:
    print("Run the comparative playtest cell first.")

## F. Reflection (500–700 words)

**Important:** This is a reflection draft to personalize after running your own three tests. Replace the bracketed observations with what you actually saw. Do not claim that RAG was faster, more accurate, or cheaper unless your run supports that conclusion.

### Reflection

Building the “Speak with the PDF” application helped me understand how Retrieval-Augmented Generation connects a language model with information from a document supplied by the user. A vanilla large language model answers from its general training and the prompt, so it may not know the specific contents of a newly uploaded course PDF. The RAG version first retrieves relevant passages from the PDF and then supplies those passages as context to the model. In my tests, the most noticeable difference was **[describe the actual difference you observed across your three questions]**. The RAG answers were more closely tied to the document, while the vanilla answers **[describe whether they were generic, incomplete, correct, or unsupported]**. Retrieval does not guarantee correctness, but it gives the model evidence that is specific to the source.

Chunking was important because embedding and searching an entire long PDF as one unit would make it difficult to identify the exact passage related to a question. I split the extracted text into chunks of approximately 800 characters with an overlap of 120 characters. The overlap helps preserve information when a sentence or explanation crosses a chunk boundary. Without overlap, a key definition might be divided between two chunks and neither retrieved passage would contain the complete idea. Very small chunks can lose context, while very large chunks can contain unrelated material and make retrieval less focused. For this assignment, the chosen size was a practical starting point; experimenting with different sizes and overlaps could improve results for PDFs with different writing styles.

Embeddings convert text into numerical vectors that represent semantic meaning. The same embedding model was used for both document chunks and the user's question, which makes their vectors comparable. Vector search then ranks chunks by cosine similarity and selects the top results. This allows the system to retrieve passages that are conceptually related to a question even when the wording is not identical. The retrieved passages become the context for the chat model. The quality of the final answer therefore depends on several stages: clean PDF extraction, sensible chunking, useful embeddings, relevant retrieval, and a prompt that tells the model to stay within the supplied context.

There are trade-offs. The first run takes time because the PDF must be parsed and every chunk must be embedded. Each question also requires an embedding request and a chat-completion request, which can add latency and API cost. A local NumPy index avoids the setup and ongoing service requirements of a hosted vector database, but it keeps the vectors in notebook memory and is less suitable for a large, persistent, multi-user application. Pinecone can provide managed storage and search, but it adds configuration and possible cost. The notebook also depends on network access and valid OpenRouter model availability.

I would extend the application in several ways. First, I would add caching or a Cache-Augmented Generation (CAG) layer so that repeated questions or previously computed document embeddings do not need to be processed again. Second, I would explore hybrid retrieval that combines semantic vector search with keyword search; this can help with exact names, technical terms, and numbers. A Knowledge-Augmented Generation (KAG) approach could add structured relationships, such as entities and connections extracted from the PDF, to improve multi-step questions. I would also add page-number metadata and citations so users can verify where each answer came from. Overall, this project showed me that RAG is not simply asking an LLM to read a PDF: it is a pipeline in which retrieval quality and source-grounding strongly influence the usefulness of generated answers.

**Before submission:** Personalize the bracketed sentence in paragraph one using your actual outputs, verify the reflection is between 500 and 700 words, and keep your three question/answer comparisons in the notebook.


---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  If the function is not found, you answer will be rejected by the system

---

## Hidden Tests (run by grader)

Hidden tests will verify that:

- `parse_pdf_to_chunks` applies overlap correctly, is deterministic, and honors size bounds.
- `embed_and_index` calls OpenAI embeddings and builds a valid index (Pinecone or local), with stable IDs and correct dimensionality/shape.
- Retrieval uses cosine similarity and returns the expected top‑k order on crafted inputs.
- `answer_query` embeds the question with the same model used for chunks, constructs a context‑bounded prompt, calls OpenAI chat completions, and returns a clean string.
- No hard‑coded secrets; keys must be read from environment.
- Notebook runs top‑to‑bottom reproducibly when keys are present.



---

## Rubric (100 pts)

- **PDF parsing & chunking (30 pts)**
  - Correct extraction and normalization (10)
  - Deterministic overlapping chunks within size bounds (15)
  - Edge cases (empty pages, weird whitespace) (5)

- **Embeddings & indexing (35 pts)**
  - Correct OpenAI embedding usage and shapes (15)
  - Proper index structure + metadata (10)
  - Stable IDs and retrievability (10)

- **Query flow & answer (25 pts)**
  - Correct cosine similarity retrieval and stable tie‑break (10)
  - Prompt enforces context‑only answering; sensible output (15)

- **Code quality & constraints (10 pts)**
  - Clean, readable, documented code (5)
  - Secure key handling; graceful error handling (5)



---

## (Optional) Pinecone/OpenAI Notes

- For Pinecone: create an index (e.g., 384 or 1536 dim depending on embedding model) and upsert vectors with metadata (`{"chunk": "...", "order": i}`).
- For local dev without Pinecone: store embeddings in a NumPy matrix and do cosine similarity. Grading is OK with either backend as long as OpenAI embeddings + chat are used.
- Keep your mapping from vector IDs to original chunk text; you’ll need it for context construction.



---

## Submission

Submit your completed notebook. Ensure it runs top‑to‑bottom without modification in a clean environment **with valid API keys set**. The grader will run hidden tests using OpenAI for embeddings and chat.


---

<h2><center> Completing your test </center></h2>

- Once you have completed your test and answered all the questions:
  
- You can open the guide using the button located on the far right of your screen.

- The button will look like this image:  
  ![](https://github.com/SasankaPandaSCIT/Automate-with-Gen-AI-Agents/blob/main/Module%203/additional_files/End_guide_button.png?raw=1)

- Clicking on this button will allow you to mark the test as complete using the button shown below:  
  ![](https://github.com/SasankaPandaSCIT/Automate-with-Gen-AI-Agents/blob/main/Module%203/additional_files/completion_button.png?raw=1)

- Please click on **Mark as Completed** to end and submit your test.

- Thank you!
